# End-to-End FMM–Segmentation Runtime Benchmark

This notebook benchmarks the complete Raw and FMM-99.5 segmentation pipelines for all ten manuscript configurations on one machine. It loads the archived checkpoints and performs inference only; no training is executed.

**Timing boundary.** Image decoding, model construction/loading, graph tracing, warm-up, result validation, and file export are excluded. Raw and FMM latency starts from already decoded OpenCV images and ends only after the segmentation probability map has been materialized on the CPU. Batch size is fixed to one.

**FMM implementation.** The FMM branch reproduces the main multi-architecture experiment: image-specific 99.5th-percentile grayscale mask, bicubic 0.6× resizing of image and mask, no mask re-binarization, OpenCV TELEA inpainting with radius 10, resize to 384×288, and an in-memory JPEG quality-95 round trip before RGB normalization.

**Environment.** Select the `Python (PR Runtime RTX 5080)` kernel. It preserves Python 3.8.18 and TensorFlow 2.9.0 and pins `keras-cv-attention-models==1.3.16` for Polyp-PVT checkpoint compatibility.

The notebook is configured with `DRY_RUN=False` for the formal three-run benchmark. Set it to `True` only when a short smoke test is needed.


In [1]:
# Thread limits must be set before importing NumPy, OpenCV, or TensorFlow.
import os

CPU_THREADS = 8
for variable in (
    "OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS",
):
    os.environ[variable] = str(CPU_THREADS)
os.environ.setdefault("SM_FRAMEWORK", "tf.keras")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "1")


'1'

In [2]:
import gc
import json
import math
import platform
import subprocess
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import cv2
import h5py
import numpy as np
import pandas as pd
import importlib.metadata as importlib_metadata
import importlib.util
import tensorflow as tf
from IPython.display import display
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.initializers import he_normal
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, AveragePooling2D, Conv2DTranspose,
    BatchNormalization, Activation, ReLU, UpSampling2D, Concatenate,
    Dropout, LeakyReLU, concatenate,
)
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2

import segmentation_models as sm

sm.set_framework("tf.keras")
if not tf.__version__.startswith("2.9."):
    raise RuntimeError(f"TensorFlow 2.9.x is required, found {tf.__version__}")
if sys.version_info[:2] != (3, 8):
    raise RuntimeError(f"Python 3.8 is required, found {sys.version.split()[0]}")
kecam_version = importlib_metadata.version("keras-cv-attention-models")
if kecam_version != "1.3.16":
    raise RuntimeError(
        "keras-cv-attention-models 1.3.16 is required, "
        f"found {kecam_version}"
    )
cv2.setNumThreads(CPU_THREADS)
tf.config.threading.set_intra_op_parallelism_threads(CPU_THREADS)
tf.config.threading.set_inter_op_parallelism_threads(1)

gpus = tf.config.list_physical_devices("GPU")
if not gpus:
    raise RuntimeError("No TensorFlow GPU was detected; CPU fallback is not allowed.")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

try:
    gpu_query = subprocess.check_output(
        [
            "nvidia-smi", "--query-gpu=name,driver_version,memory.total",
            "--format=csv,noheader,nounits",
        ],
        text=True,
    ).strip()
except Exception as error:
    gpu_query = f"Unavailable: {error}"
if "RTX 5080" not in gpu_query:
    warnings.warn(f"Expected RTX 5080, detected: {gpu_query}")

print("Python:", sys.version.split()[0])
print("TensorFlow:", tf.__version__)
print("OpenCV:", cv2.__version__)
print("keras-cv-attention-models:", kecam_version)
print("GPU:", gpu_query)


Segmentation Models: using `tf.keras` framework.
Python: 3.8.18
TensorFlow: 2.9.0
OpenCV: 4.9.0
keras-cv-attention-models: 1.3.16
GPU: NVIDIA GeForce RTX 5080, 595.58.03, 16303


2026-11-09 16:39:42.569377: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1943] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.


In [3]:
# =========================
# User-facing configuration
# =========================
PROJECT_ROOT = Path("/home/wujieyu/jupyterlab/tensorflow/supplement/PR-R2")
RUNTIME_DIR = PROJECT_ROOT / "code" / "Supplementary_Code" / "03_runtime_analysis"
EXISTING_MANIFEST = (
    RUNTIME_DIR / "outputs_highlight_removal_runtime" / "runtime_input_manifest.csv"
)
OUTPUT_ROOT = RUNTIME_DIR / "outputs_end_to_end_runtime"

DRY_RUN = False
DRY_RUN_IMAGES = 3
RUNS = 3
WARMUP_IMAGES = 30
BATCH_SIZE = 1
PERCENTILE = 99.5
INPAINT_RADIUS = 10
JPEG_QUALITY = 95
IMAGE_HEIGHT = 288
IMAGE_WIDTH = 384
IMAGE_SIZE = (IMAGE_WIDTH, IMAGE_HEIGHT)
EXPECTED_IMAGES = 2987
RUN_TRAINING = False

ALL_MODELS = [
    "U-Net++", "U-Net+V3", "FPN+V3", "LinkNet+V3", "PSPNet+V3",
    "SegNet-Simple", "SegNet", "PraNet", "TransUNet", "Polyp-PVT",
]
RUN_MODELS = ALL_MODELS.copy()

MODEL_SPECS = {
    "U-Net++": {"directory": "U-net++", "weight_stem": "U-net++"},
    "U-Net+V3": {"directory": "U-net+V3", "weight_stem": "U-net+V3"},
    "FPN+V3": {"directory": "FPN+V3", "weight_stem": "FPN+V3"},
    "LinkNet+V3": {"directory": "Linknet+V3", "weight_stem": "Linknet+V3"},
    "PSPNet+V3": {"directory": "PSPnet+V3", "weight_stem": "PSPnet+V3"},
    # Manuscript SegNet-Simple corresponds to the archived SegNet directory.
    "SegNet-Simple": {"directory": "SegNet", "weight_stem": "SegNet"},
    # Manuscript SegNet corresponds to the archived SegNet-VGG directory.
    "SegNet": {"directory": "SegNet-VGG", "weight_stem": "SegNet-VGG"},
    "PraNet": {"directory": "PraNet", "weight_stem": "PraNet"},
    "TransUNet": {"directory": "TransUNet", "weight_stem": "TransUNet"},
    "Polyp-PVT": {"directory": "Polyp-PVT", "weight_stem": "Polyp-PVT"},
}

unknown_models = sorted(set(RUN_MODELS) - set(ALL_MODELS))
if unknown_models:
    raise ValueError(f"Unknown RUN_MODELS entries: {unknown_models}")
if BATCH_SIZE != 1:
    raise ValueError("This benchmark requires BATCH_SIZE=1.")

EFFECTIVE_RUNS = 1 if DRY_RUN else RUNS
EFFECTIVE_WARMUP = 1 if DRY_RUN else WARMUP_IMAGES
OUTPUT_DIR = OUTPUT_ROOT / "dry_run" if DRY_RUN else OUTPUT_ROOT
PARTIAL_DIR = OUTPUT_DIR / "per_model"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PARTIAL_DIR.mkdir(parents=True, exist_ok=True)

print("Mode:", "DRY RUN" if DRY_RUN else "FORMAL")
print("Models:", RUN_MODELS)
print("Output:", OUTPUT_DIR)


Mode: FORMAL
Models: ['U-Net++', 'U-Net+V3', 'FPN+V3', 'LinkNet+V3', 'PSPNet+V3', 'SegNet-Simple', 'SegNet', 'PraNet', 'TransUNet', 'Polyp-PVT']
Output: /home/wujieyu/jupyterlab/tensorflow/supplement/PR-R2/code/Supplementary_Code/03_runtime_analysis/outputs_end_to_end_runtime


In [4]:
# =========================
# Input manifest and weights
# =========================
TRAIN_IMAGE_DIR = PROJECT_ROOT / "Data" / "Train" / "img"
WEIGHT_ROOT = PROJECT_ROOT / "best weight" / "2.different mode"

if not EXISTING_MANIFEST.is_file():
    raise FileNotFoundError(f"Existing runtime manifest not found: {EXISTING_MANIFEST}")
manifest = pd.read_csv(EXISTING_MANIFEST)
required_manifest_columns = {"image_name", "file_size_bytes", "sha256"}
if not required_manifest_columns.issubset(manifest.columns):
    raise ValueError(f"Unexpected manifest columns: {manifest.columns.tolist()}")
if len(manifest) != EXPECTED_IMAGES:
    raise ValueError(f"Expected {EXPECTED_IMAGES} images, found {len(manifest)}")
if manifest["image_name"].duplicated().any():
    raise ValueError("Input manifest contains duplicate image names.")

manifest["image_path"] = manifest["image_name"].map(lambda name: TRAIN_IMAGE_DIR / name)
missing_images = [str(path) for path in manifest["image_path"] if not path.is_file()]
if missing_images:
    raise FileNotFoundError(f"Missing {len(missing_images)} images, e.g. {missing_images[:3]}")
if DRY_RUN:
    active_manifest = manifest.iloc[:DRY_RUN_IMAGES].copy()
else:
    active_manifest = manifest.copy()

def resolve_weight_paths(model_name):
    spec = MODEL_SPECS[model_name]
    model_dir = WEIGHT_ROOT / spec["directory"]
    stem = spec["weight_stem"]
    raw = model_dir / "img" / f"{stem}_img_bestweight.h5"
    fmm = model_dir / "FMM+percentile=99.5" / f"{stem}_99.5+FMM_bestweight.h5"
    return raw, fmm

weight_rows = []
for model_name in ALL_MODELS:
    raw_weight, fmm_weight = resolve_weight_paths(model_name)
    for condition, path in (("Raw", raw_weight), ("FMM-99.5", fmm_weight)):
        if not path.is_file():
            raise FileNotFoundError(f"Missing {model_name}/{condition} weight: {path}")
        weight_rows.append(
            {
                "model": model_name,
                "condition": condition,
                "weight_path": str(path),
                "weight_size_bytes": path.stat().st_size,
            }
        )
WEIGHT_MANIFEST = pd.DataFrame(weight_rows)
print(f"Validated {len(manifest)} input images and {len(WEIGHT_MANIFEST)} checkpoints.")
display(WEIGHT_MANIFEST)


Validated 2987 input images and 20 checkpoints.


,model,condition,weight_path,weight_size_bytes
0,U-Net++,Raw,/home/wujieyu/jupyterlab/tensorflow/supplement...,36477040
1,U-Net++,FMM-99.5,/home/wujieyu/jupyterlab/tensorflow/supplement...,36477040
2,U-Net+V3,Raw,/home/wujieyu/jupyterlab/tensorflow/supplement...,120525312
3,U-Net+V3,FMM-99.5,/home/wujieyu/jupyterlab/tensorflow/supplement...,120525312
4,FPN+V3,Raw,/home/wujieyu/jupyterlab/tensorflow/supplement...,100911104
5,FPN+V3,FMM-99.5,/home/wujieyu/jupyterlab/tensorflow/supplement...,100911104
6,LinkNet+V3,Raw,/home/wujieyu/jupyterlab/tensorflow/supplement...,105905104
7,LinkNet+V3,FMM-99.5,/home/wujieyu/jupyterlab/tensorflow/supplement...,105905104
8,PSPNet+V3,Raw,/home/wujieyu/jupyterlab/tensorflow/supplement...,11421096
9,PSPNet+V3,FMM-99.5,/home/wujieyu/jupyterlab/tensorflow/supplement...,11421096


In [5]:
# =========================
# Raw and main-experiment FMM preprocessing
# =========================
def decode_image_pair(image_path):
    color = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    gray = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if color is None or gray is None:
        raise ValueError(f"Unable to decode image: {image_path}")
    return color, gray


def preprocess_raw(decoded_bgr):
    image = cv2.resize(decoded_bgr, IMAGE_SIZE)
    image = image[..., ::-1]
    return np.ascontiguousarray(image, dtype=np.float32) / 255.0


def preprocess_fmm(decoded_bgr, decoded_gray, return_mask=False):
    threshold = np.percentile(decoded_gray, PERCENTILE)
    _, binary_mask = cv2.threshold(
        decoded_gray, threshold, 255, cv2.THRESH_BINARY
    )
    resized_image = cv2.resize(
        decoded_bgr, None, fx=0.6, fy=0.6, interpolation=cv2.INTER_CUBIC
    )
    # Deliberately no re-binarization: this reproduces the main experiment.
    resized_mask = cv2.resize(
        binary_mask, None, fx=0.6, fy=0.6, interpolation=cv2.INTER_CUBIC
    )
    restored = cv2.inpaint(
        resized_image, resized_mask, INPAINT_RADIUS, cv2.INPAINT_TELEA
    )
    restored = cv2.resize(restored, IMAGE_SIZE)
    ok, encoded = cv2.imencode(
        ".jpg", restored, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY]
    )
    if not ok:
        raise RuntimeError("In-memory JPEG encoding failed.")
    restored = cv2.imdecode(encoded, cv2.IMREAD_COLOR)
    if restored is None:
        raise RuntimeError("In-memory JPEG decoding failed.")
    restored = restored[..., ::-1]
    normalized = np.ascontiguousarray(restored, dtype=np.float32) / 255.0
    if return_mask:
        return normalized, binary_mask, resized_mask, float(threshold)
    return normalized


def archived_reference_fmm(image_path):
    source = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    gray = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    threshold = np.percentile(gray, PERCENTILE)
    _, mask = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)
    source = cv2.resize(
        source, None, fx=0.6, fy=0.6, interpolation=cv2.INTER_CUBIC
    )
    mask = cv2.resize(
        mask, None, fx=0.6, fy=0.6, interpolation=cv2.INTER_CUBIC
    )
    restored = cv2.inpaint(source, mask, INPAINT_RADIUS, cv2.INPAINT_TELEA)
    restored = cv2.resize(restored, IMAGE_SIZE)
    ok, encoded = cv2.imencode(
        ".jpg", restored, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY]
    )
    if not ok:
        raise RuntimeError(f"Reference JPEG encoding failed: {image_path}")
    restored = cv2.imdecode(encoded, cv2.IMREAD_COLOR)[..., ::-1]
    return restored.astype(np.float32) / 255.0


# Pixelwise parity check against the archived multi-architecture implementation.
parity_sample = manifest.iloc[np.linspace(0, len(manifest) - 1, 5, dtype=int)]
interpolation_values = set()
for image_path in parity_sample["image_path"]:
    color, gray = decode_image_pair(image_path)
    candidate, binary_mask, resized_mask, _ = preprocess_fmm(
        color, gray, return_mask=True
    )
    reference = archived_reference_fmm(image_path)
    if not np.array_equal(candidate, reference):
        max_difference = float(np.max(np.abs(candidate - reference)))
        raise AssertionError(
            f"FMM parity failed for {image_path}; max difference={max_difference}"
        )
    if set(np.unique(binary_mask)) - {0, 255}:
        raise AssertionError("Pre-resize highlight mask is not binary.")
    interpolation_values.update(np.unique(resized_mask).tolist())

if not (interpolation_values - {0, 255}):
    warnings.warn("Parity sample did not expose intermediate bicubic mask values.")
print("FMM parity passed for 5 images.")
print("Resized mask contains non-binary values and is not re-thresholded:",
      bool(interpolation_values - {0, 255}))


FMM parity passed for 5 images.
Resized mask contains non-binary values and is not re-thresholded: True


In [6]:
# =========================
# Strict HDF5 checkpoint loading
# =========================
def _keras3_h5_layout(h5_file):
    return (
        "layers" in h5_file
        and "vars" in h5_file
        and "layer_names" not in h5_file.attrs
    )


def _load_keras3_h5_weights(model, weight_path):
    loaded_names = set()
    with h5py.File(weight_path, "r") as h5_file:
        layer_store = h5_file["layers"]
        file_weighted_names = {
            name
            for name, group in layer_store.items()
            if "vars" in group and len(group["vars"].keys())
        }
        for layer in model.layers:
            expected = layer.get_weights()
            if not expected:
                continue
            if layer.name not in layer_store:
                raise ValueError(f"Checkpoint is missing layer {layer.name!r}")
            variable_store = layer_store[layer.name].get("vars")
            if variable_store is None:
                raise ValueError(f"Checkpoint layer {layer.name!r} has no vars group")
            keys = sorted(variable_store.keys(), key=int)
            values = [np.asarray(variable_store[key]) for key in keys]
            if len(values) != len(expected):
                raise ValueError(
                    f"Weight count mismatch for {layer.name!r}: "
                    f"file={len(values)}, model={len(expected)}"
                )
            for index, (value, target) in enumerate(zip(values, expected)):
                if value.shape != target.shape:
                    raise ValueError(
                        f"Shape mismatch for {layer.name!r}[{index}]: "
                        f"file={value.shape}, model={target.shape}"
                    )
            layer.set_weights(values)
            loaded_names.add(layer.name)
        unused = file_weighted_names - loaded_names
        if unused:
            raise ValueError(
                "Weighted checkpoint layers were not loaded: "
                + ", ".join(sorted(unused)[:20])
            )
    if not loaded_names:
        raise ValueError("No Keras 3 HDF5 layers were loaded.")
    return len(loaded_names)


def load_selected_weights(model, weight_path, model_name):
    try:
        with h5py.File(weight_path, "r") as h5_file:
            keras3_layout = _keras3_h5_layout(h5_file)
        if keras3_layout:
            count = _load_keras3_h5_weights(model, weight_path)
            print(f"{model_name}: strictly loaded {count} Keras 3 HDF5 layers")
        else:
            model.load_weights(str(weight_path))
            print(f"{model_name}: strictly loaded legacy HDF5 weights by topology")
    except Exception as error:
        raise RuntimeError(
            f"Unable to load the complete {model_name} checkpoint: {weight_path}"
        ) from error


In [7]:
# =========================
# Exact inference-only model definitions copied from the source notebooks
# =========================
def build_u_net_plus_plus():
    # Architecture copied verbatim from U-net++-Raw-vs-FMM-Train-Inference.ipynb.
    def standard_unit(input_tensor, stage, nb_filter, kernel_size=3):
        """原始U-Net++的标准卷积单元（包含BN和ReLU）"""
        x = Conv2D(nb_filter, (kernel_size, kernel_size), 
                   activation='relu', 
                   name='conv'+stage+'_1', 
                   kernel_initializer='he_normal', 
                   padding='same', 
                   kernel_regularizer=l2(1e-4))(input_tensor)
        x = BatchNormalization(name='bn'+stage)(x)
        x = Conv2D(nb_filter, (kernel_size, kernel_size), 
                   activation='relu',
                   name='conv'+stage+'_2', 
                   kernel_initializer='he_normal', 
                   padding='same', 
                   kernel_regularizer=l2(1e-4))(x)
        x = BatchNormalization(name='bn'+stage+'_2')(x)
        return x

    def unetpp(num_class=1):  # 论文默认启用深度监督
        print('原始U-Net++（2018 CVPR版本）')
        nb_filter = [32, 64, 128, 256, 512]

        # 输入层
        img_input = Input(shape=(288, 384, 3), name='main_input')

        # ----------- 编码器路径 -----------
        # Block 1 (X00)
        x1_1 = standard_unit(img_input, stage='11', nb_filter=nb_filter[0])
        pool1 = MaxPooling2D((2, 2), name='pool1')(x1_1)

        # Block 2 (X10)
        x2_1 = standard_unit(pool1, stage='21', nb_filter=nb_filter[1])
        pool2 = MaxPooling2D((2, 2), name='pool2')(x2_1)

        # Block 3 (X20)
        x3_1 = standard_unit(pool2, stage='31', nb_filter=nb_filter[2])
        pool3 = MaxPooling2D((2, 2), name='pool3')(x3_1)

        # Block 4 (X30)
        x4_1 = standard_unit(pool3, stage='41', nb_filter=nb_filter[3])
        pool4 = MaxPooling2D((2, 2), name='pool4')(x4_1)

        # ----------- 瓶颈层 -----------
        x5_1 = standard_unit(pool4, stage='51', nb_filter=nb_filter[4])

        # ----------- 解码器路径（密集跳跃连接）-----------
        # Block 5 (X01)
        x1_2 = Conv2DTranspose(nb_filter[0], (2, 2), strides=(2, 2), name='up12', padding='same')(x2_1)
        x1_2 = concatenate([x1_2, x1_1], name='merge12')
        x1_2 = standard_unit(x1_2, stage='12', nb_filter=nb_filter[0])

        # Block 6 (X11)
        x2_2 = Conv2DTranspose(nb_filter[1], (2, 2), strides=(2, 2), name='up22', padding='same')(x3_1)
        x2_2 = concatenate([x2_2, x2_1], name='merge22')
        x2_2 = standard_unit(x2_2, stage='22', nb_filter=nb_filter[1])

        # Block 7 (X02)
        x1_3 = Conv2DTranspose(nb_filter[0], (2, 2), strides=(2, 2), name='up13', padding='same')(x2_2)
        x1_3 = concatenate([x1_3, x1_1, x1_2], name='merge13')
        x1_3 = standard_unit(x1_3, stage='13', nb_filter=nb_filter[0])

        # 继续构建完整结构...
        # Block 8 (X21)
        x3_2 = Conv2DTranspose(nb_filter[2], (2, 2), strides=(2, 2), name='up32', padding='same')(x4_1)
        x3_2 = concatenate([x3_2, x3_1], name='merge32')
        x3_2 = standard_unit(x3_2, stage='32', nb_filter=nb_filter[2])

        # Block 9 (X12)
        x2_3 = Conv2DTranspose(nb_filter[1], (2, 2), strides=(2, 2), name='up23', padding='same')(x3_2)
        x2_3 = concatenate([x2_3, x2_1, x2_2], name='merge23')
        x2_3 = standard_unit(x2_3, stage='23', nb_filter=nb_filter[1])

        # Block 10 (X03)
        x1_4 = Conv2DTranspose(nb_filter[0], (2, 2), strides=(2, 2), name='up14', padding='same')(x2_3)
        x1_4 = concatenate([x1_4, x1_1, x1_2, x1_3], name='merge14')
        x1_4 = standard_unit(x1_4, stage='14', nb_filter=nb_filter[0])

        # ----------- 最深层的上采样路径 -----------
        # Block 11 (X31)
        x4_2 = Conv2DTranspose(nb_filter[3], (2, 2), strides=(2, 2), name='up42', padding='same')(x5_1)
        x4_2 = concatenate([x4_2, x4_1], name='merge42')
        x4_2 = standard_unit(x4_2, stage='42', nb_filter=nb_filter[3])

        # Block 12 (X22)
        x3_3 = Conv2DTranspose(nb_filter[2], (2, 2), strides=(2, 2), name='up33', padding='same')(x4_2)
        x3_3 = concatenate([x3_3, x3_1, x3_2], name='merge33')
        x3_3 = standard_unit(x3_3, stage='33', nb_filter=nb_filter[2])

        # Block 13 (X13)
        x2_4 = Conv2DTranspose(nb_filter[1], (2, 2), strides=(2, 2), name='up24', padding='same')(x3_3)
        x2_4 = concatenate([x2_4, x2_1, x2_2, x2_3], name='merge24')
        x2_4 = standard_unit(x2_4, stage='24', nb_filter=nb_filter[1])

        # Block 14 (X04)
        x1_5 = Conv2DTranspose(nb_filter[0], (2, 2), strides=(2, 2), name='up15', padding='same')(x2_4)
        x1_5 = concatenate([x1_5, x1_1, x1_2, x1_3, x1_4], name='merge15')
        x1_5 = standard_unit(x1_5, stage='15', nb_filter=nb_filter[0])

        # ----------- 输出层 -----------
        output1 = Conv2D(num_class, (1, 1), activation='sigmoid', name='output1')(x1_2)
        output2 = Conv2D(num_class, (1, 1), activation='sigmoid', name='output2')(x1_3)
        output3 = Conv2D(num_class, (1, 1), activation='sigmoid', name='output3')(x1_4)
        output4 = Conv2D(num_class, (1, 1), activation='sigmoid', name='output4')(x1_5)

    #     model = Model(inputs=img_input, outputs=[output1, output2, output3, output4])
        model = Model(inputs=img_input, outputs=output4)
        return model

    def build_model():
        return unetpp(num_class=1)
    return build_model()



def build_u_net_v3():
    # Architecture copied verbatim from U-net+V3-Raw-vs-FMM-Train-Inference.ipynb.
    def build_model():
        BACKBONE = 'inceptionv3'
        return sm.Unet(
            BACKBONE, encoder_weights='imagenet' if RUN_TRAINING else None
        )
    return build_model()



def build_fpn_v3():
    # Architecture copied verbatim from FPN+V3-Raw-vs-FMM-Train-Inference.ipynb.
    def build_model():
        BACKBONE = 'inceptionv3'
        return sm.FPN(
            BACKBONE, classes=1, activation='sigmoid',
            encoder_weights='imagenet' if RUN_TRAINING else None
        )
    return build_model()



def build_linknet_v3():
    # Architecture copied verbatim from Linknet+V3-Raw-vs-FMM-Train-Inference.ipynb.
    def build_model():
        BACKBONE = 'inceptionv3'
        return sm.Linknet(
            BACKBONE, classes=1, activation='sigmoid',
            encoder_weights='imagenet' if RUN_TRAINING else None
        )
    return build_model()



def build_pspnet_v3():
    # Architecture copied verbatim from PSPnet+V3-Raw-vs-FMM-Train-Inference.ipynb.
    def build_model():
        BACKBONE = 'inceptionv3'
        return sm.PSPNet(
            BACKBONE,
            input_shape=(288, 384, 3),
            classes=1,
            activation='sigmoid',
            encoder_weights='imagenet' if RUN_TRAINING else None
        )
    return build_model()



def build_segnet_simple():
    # Architecture copied verbatim from SegNet-simple-Raw-vs-FMM-Train-Inference.ipynb.
    def SegNet(input_shape=(224,224,3), n_classes=1):
        inputs = layers.Input(shape=input_shape)

        # ---------- Encoder ----------
        # Block 1
        x = layers.Conv2D(64, (3,3), padding="same", activation="relu")(inputs)
        x = layers.Conv2D(64, (3,3), padding="same", activation="relu")(x)
        p1 = layers.MaxPooling2D((2,2))(x)

        # Block 2
        x = layers.Conv2D(128, (3,3), padding="same", activation="relu")(p1)
        x = layers.Conv2D(128, (3,3), padding="same", activation="relu")(x)
        p2 = layers.MaxPooling2D((2,2))(x)

        # Block 3
        x = layers.Conv2D(256, (3,3), padding="same", activation="relu")(p2)
        x = layers.Conv2D(256, (3,3), padding="same", activation="relu")(x)
        x = layers.Conv2D(256, (3,3), padding="same", activation="relu")(x)
        p3 = layers.MaxPooling2D((2,2))(x)

        # Block 4
        x = layers.Conv2D(512, (3,3), padding="same", activation="relu")(p3)
        x = layers.Conv2D(512, (3,3), padding="same", activation="relu")(x)
        x = layers.Conv2D(512, (3,3), padding="same", activation="relu")(x)
        p4 = layers.MaxPooling2D((2,2))(x)

        # Block 5
        x = layers.Conv2D(512, (3,3), padding="same", activation="relu")(p4)
        x = layers.Conv2D(512, (3,3), padding="same", activation="relu")(x)
        x = layers.Conv2D(512, (3,3), padding="same", activation="relu")(x)
        p5 = layers.MaxPooling2D((2,2))(x)

        # ---------- Decoder ----------
        # Block 5D
        x = layers.UpSampling2D((2,2))(p5)
        x = layers.Conv2D(512, (3,3), padding="same", activation="relu")(x)
        x = layers.Conv2D(512, (3,3), padding="same", activation="relu")(x)
        x = layers.Conv2D(512, (3,3), padding="same", activation="relu")(x)

        # Block 4D
        x = layers.UpSampling2D((2,2))(x)
        x = layers.Conv2D(512, (3,3), padding="same", activation="relu")(x)
        x = layers.Conv2D(512, (3,3), padding="same", activation="relu")(x)
        x = layers.Conv2D(256, (3,3), padding="same", activation="relu")(x)

        # Block 3D
        x = layers.UpSampling2D((2,2))(x)
        x = layers.Conv2D(256, (3,3), padding="same", activation="relu")(x)
        x = layers.Conv2D(256, (3,3), padding="same", activation="relu")(x)
        x = layers.Conv2D(128, (3,3), padding="same", activation="relu")(x)

        # Block 2D
        x = layers.UpSampling2D((2,2))(x)
        x = layers.Conv2D(128, (3,3), padding="same", activation="relu")(x)
        x = layers.Conv2D(64, (3,3), padding="same", activation="relu")(x)

        # Block 1D
        x = layers.UpSampling2D((2,2))(x)
        x = layers.Conv2D(64, (3,3), padding="same", activation="relu")(x)

        # 输出层
        outputs = layers.Conv2D(n_classes, (1,1), activation="sigmoid")(x)

        model = models.Model(inputs=inputs, outputs=outputs)
        return model


    def build_model():
        return SegNet(input_shape=(288, 384, 3), n_classes=1)
    return build_model()



def build_segnet():
    # Architecture copied verbatim from SegNet-VGG-Raw-vs-FMM-Train-Inference.ipynb.
    def VggSegNet(input_shape, num_classes=1):  # 默认输出单通道（黑白掩膜）
        inputs = layers.Input(shape=input_shape)

        # ---------------------------- 编码器部分 ----------------------------
        # Block 1 (2卷积层)
        conv_1 = layers.Conv2D(64, (3,3), padding='same')(inputs)
        conv_1 = layers.BatchNormalization()(conv_1)
        conv_1 = layers.Activation('relu')(conv_1)
        conv_2 = layers.Conv2D(64, (3,3), padding='same')(conv_1)
        conv_2 = layers.BatchNormalization()(conv_2)
        conv_2 = layers.Activation('relu')(conv_2)
        pool_1 = layers.MaxPooling2D((2,2))(conv_2)  # 144x192

        # Block 2 (2卷积层)
        conv_3 = layers.Conv2D(128, (3,3), padding='same')(pool_1)
        conv_3 = layers.BatchNormalization()(conv_3)
        conv_3 = layers.Activation('relu')(conv_3)
        conv_4 = layers.Conv2D(128, (3,3), padding='same')(conv_3)
        conv_4 = layers.BatchNormalization()(conv_4)
        conv_4 = layers.Activation('relu')(conv_4)
        pool_2 = layers.MaxPooling2D((2,2))(conv_4)  # 72x96

        # Block 3 (3卷积层)
        conv_5 = layers.Conv2D(256, (3,3), padding='same')(pool_2)
        conv_5 = layers.BatchNormalization()(conv_5)
        conv_5 = layers.Activation('relu')(conv_5)
        conv_6 = layers.Conv2D(256, (3,3), padding='same')(conv_5)
        conv_6 = layers.BatchNormalization()(conv_6)
        conv_6 = layers.Activation('relu')(conv_6)
        conv_7 = layers.Conv2D(256, (3,3), padding='same')(conv_6)
        conv_7 = layers.BatchNormalization()(conv_7)
        conv_7 = layers.Activation('relu')(conv_7)
        pool_3 = layers.MaxPooling2D((2,2))(conv_7)  # 36x48

        # Block 4 (3卷积层)
        conv_8 = layers.Conv2D(512, (3,3), padding='same')(pool_3)
        conv_8 = layers.BatchNormalization()(conv_8)
        conv_8 = layers.Activation('relu')(conv_8)
        conv_9 = layers.Conv2D(512, (3,3), padding='same')(conv_8)
        conv_9 = layers.BatchNormalization()(conv_9)
        conv_9 = layers.Activation('relu')(conv_9)
        conv_10 = layers.Conv2D(512, (3,3), padding='same')(conv_9)
        conv_10 = layers.BatchNormalization()(conv_10)
        conv_10 = layers.Activation('relu')(conv_10)
        pool_4 = layers.MaxPooling2D((2,2))(conv_10)  # 18x24

        # Block 5 (3卷积层)
        conv_11 = layers.Conv2D(512, (3,3), padding='same')(pool_4)
        conv_11 = layers.BatchNormalization()(conv_11)
        conv_11 = layers.Activation('relu')(conv_11)
        conv_12 = layers.Conv2D(512, (3,3), padding='same')(conv_11)
        conv_12 = layers.BatchNormalization()(conv_12)
        conv_12 = layers.Activation('relu')(conv_12)
        conv_13 = layers.Conv2D(512, (3,3), padding='same')(conv_12)
        conv_13 = layers.BatchNormalization()(conv_13)
        conv_13 = layers.Activation('relu')(conv_13)
        pool_5 = layers.MaxPooling2D((2,2))(conv_13)  # 9x12

        # ---------------------------- 解码器部分 ----------------------------
        # Block 1解码 (3卷积层，对应Block5编码)
        unpool_1 = layers.UpSampling2D((2,2))(pool_5)  # 18x24
        conv_14 = layers.Conv2D(512, (3,3), padding='same')(unpool_1)
        conv_14 = layers.BatchNormalization()(conv_14)
        conv_14 = layers.Activation('relu')(conv_14)
        conv_15 = layers.Conv2D(512, (3,3), padding='same')(conv_14)
        conv_15 = layers.BatchNormalization()(conv_15)
        conv_15 = layers.Activation('relu')(conv_15)
        conv_16 = layers.Conv2D(512, (3,3), padding='same')(conv_15)
        conv_16 = layers.BatchNormalization()(conv_16)
        conv_16 = layers.Activation('relu')(conv_16)

        # Block 2解码 (3卷积层，对应Block4编码)
        unpool_2 = layers.UpSampling2D((2,2))(conv_16)  # 36x48
        conv_17 = layers.Conv2D(512, (3,3), padding='same')(unpool_2)
        conv_17 = layers.BatchNormalization()(conv_17)
        conv_17 = layers.Activation('relu')(conv_17)
        conv_18 = layers.Conv2D(512, (3,3), padding='same')(conv_17)
        conv_18 = layers.BatchNormalization()(conv_18)
        conv_18 = layers.Activation('relu')(conv_18)
        conv_19 = layers.Conv2D(256, (3,3), padding='same')(conv_18)
        conv_19 = layers.BatchNormalization()(conv_19)
        conv_19 = layers.Activation('relu')(conv_19)

        # Block 3解码 (3卷积层，对应Block3编码)
        unpool_3 = layers.UpSampling2D((2,2))(conv_19)  # 72x96
        conv_20 = layers.Conv2D(256, (3,3), padding='same')(unpool_3)
        conv_20 = layers.BatchNormalization()(conv_20)
        conv_20 = layers.Activation('relu')(conv_20)
        conv_21 = layers.Conv2D(256, (3,3), padding='same')(conv_20)
        conv_21 = layers.BatchNormalization()(conv_21)
        conv_21 = layers.Activation('relu')(conv_21)
        conv_22 = layers.Conv2D(128, (3,3), padding='same')(conv_21)
        conv_22 = layers.BatchNormalization()(conv_22)
        conv_22 = layers.Activation('relu')(conv_22)

        # Block 4解码 (2卷积层，对应Block2编码)
        unpool_4 = layers.UpSampling2D((2,2))(conv_22)  # 144x192
        conv_23 = layers.Conv2D(128, (3,3), padding='same')(unpool_4)
        conv_23 = layers.BatchNormalization()(conv_23)
        conv_23 = layers.Activation('relu')(conv_23)
        conv_24 = layers.Conv2D(64, (3,3), padding='same')(conv_23)
        conv_24 = layers.BatchNormalization()(conv_24)
        conv_24 = layers.Activation('relu')(conv_24)

        # Block 5解码 (2卷积层，对应Block1编码)
        unpool_5 = layers.UpSampling2D((2,2))(conv_24)  # 288x384
        conv_25 = layers.Conv2D(64, (3,3), padding='same')(unpool_5)
        conv_25 = layers.BatchNormalization()(conv_25)
        conv_25 = layers.Activation('relu')(conv_25)
        conv_26 = layers.Conv2D(64, (3,3), padding='same')(conv_25)  # 新增对称卷积层
        conv_26 = layers.BatchNormalization()(conv_26)
        conv_26 = layers.Activation('relu')(conv_26)

        # 输出层（移除BN，直接Sigmoid）
        outputs = layers.Conv2D(num_classes, (1,1), padding='valid', activation='sigmoid')(conv_26)

        model = models.Model(inputs=inputs, outputs=outputs)
        return model

    def build_model():
        return VggSegNet(input_shape=(288, 384, 3), num_classes=1)
    return build_model()



def build_pranet_model():
    # Architecture copied verbatim from PraNet-Raw-vs-FMM-Train-Inference.ipynb.
    # 1. 定义 RFB_modified 模块
    class RFB_modified(tf.keras.layers.Layer):
        def __init__(self, in_channels, out_channels):
            super().__init__()
            # 四个并行分支
            self.branch0 = models.Sequential([
                Conv2D(out_channels, 1, padding='same'),
                BatchNormalization(), ReLU()
            ])
            self.branch1 = models.Sequential([
                Conv2D(out_channels, 1, padding='same'),
                BatchNormalization(), ReLU(),
                Conv2D(out_channels, (1,3), padding='same'),
                BatchNormalization(), ReLU(),
                Conv2D(out_channels, (3,1), padding='same'),
                BatchNormalization(), ReLU(),
                Conv2D(out_channels, 3, padding='same', dilation_rate=3),
                BatchNormalization(), ReLU()
            ])
            self.branch2 = models.Sequential([
                Conv2D(out_channels, 1, padding='same'),
                BatchNormalization(), ReLU(),
                Conv2D(out_channels, (1,5), padding='same'),
                BatchNormalization(), ReLU(),
                Conv2D(out_channels, (5,1), padding='same'),
                BatchNormalization(), ReLU(),
                Conv2D(out_channels, 3, padding='same', dilation_rate=5),
                BatchNormalization(), ReLU()
            ])
            self.branch3 = models.Sequential([
                Conv2D(out_channels, 1, padding='same'),
                BatchNormalization(), ReLU(),
                Conv2D(out_channels, (1,7), padding='same'),
                BatchNormalization(), ReLU(),
                Conv2D(out_channels, (7,1), padding='same'),
                BatchNormalization(), ReLU(),
                Conv2D(out_channels, 3, padding='same', dilation_rate=7),
                BatchNormalization(), ReLU()
            ])
            # 融合与残差
            self.conv_cat = Conv2D(out_channels, 3, padding='same')
            self.bn_cat   = BatchNormalization()
            self.relu_cat = ReLU()
            self.conv_res = Conv2D(out_channels, 1, padding='same')
            self.bn_res   = BatchNormalization()
            self.relu     = ReLU()

        def call(self, x):
            x0 = self.branch0(x)
            x1 = self.branch1(x)
            x2 = self.branch2(x)
            x3 = self.branch3(x)
            x_cat = Concatenate()([x0, x1, x2, x3])
            x_cat = self.conv_cat(x_cat)
            x_cat = self.bn_cat(x_cat)
            x_cat = self.relu_cat(x_cat)
            res   = self.conv_res(x)
            res   = self.bn_res(res)
            return self.relu(x_cat + res)

    # 2. 定义 Aggregation 模块
    class Aggregation(tf.keras.layers.Layer):
        def __init__(self, channel):
            super().__init__()
            self.conv_upsample1 = Conv2D(channel, 3, padding='same')
            self.conv_upsample2 = Conv2D(channel, 3, padding='same')
            self.conv_upsample3 = Conv2D(channel, 3, padding='same')
            self.conv_concat2   = Conv2D(channel, 3, padding='same')
            self.conv_concat3   = Conv2D(channel, 3, padding='same')
            self.conv4          = Conv2D(channel, 3, padding='same')
            self.conv5          = Conv2D(1,       1, padding='same')

        def call(self, x1, x2, x3):
            x2_1 = self.conv_upsample1(UpSampling2D(2)(x1)) * x2
            x3_1 = self.conv_upsample2(UpSampling2D(2)(x2_1)) * \
                   self.conv_upsample3(UpSampling2D(4)(x1)) * x3
            x2_2 = Concatenate()([x2_1, UpSampling2D(2)(x1)])
            x2_2 = self.conv_concat2(x2_2)
            x3_2 = Concatenate()([x3_1, UpSampling2D(2)(x2_2)])
            x3_2 = self.conv_concat3(x3_2)
            x    = self.conv4(x3_2)
            return self.conv5(x)

    # 3. 定义 ReverseAttention 模块
    class ReverseAttention(tf.keras.layers.Layer):
        def __init__(self, in_channels):
            super().__init__()
            self.conv1 = Conv2D(in_channels, 3, padding='same')
            self.conv2 = Conv2D(in_channels, 3, padding='same')
            self.conv3 = Conv2D(1,           3, padding='same')

        def call(self, x, y):
            # 将粗预测 y 缩放到 x 的尺寸
            y_resized = tf.image.resize(y, tf.shape(x)[1:3], method='bilinear')
            reverse   = 1 - tf.sigmoid(y_resized)
            x_att     = x * reverse
            x_att     = self.conv1(x_att)
            x_att     = self.conv2(x_att)
            x_att     = self.conv3(x_att)
            return x_att + y_resized

    # 4. 构建 ResNet50 主干
    def build_backbone(input_shape):
        base = tf.keras.applications.ResNet50(
            include_top=False,
            weights='imagenet' if RUN_TRAINING else None,
            input_shape=input_shape
        )
        # C2,C3,C4,C5 分别对应下采样 4,8,16,32 倍之后的特征图
        return models.Model(
            inputs=base.input,
            outputs=[
                base.get_layer('conv2_block3_out').output,
                base.get_layer('conv3_block4_out').output,
                base.get_layer('conv4_block6_out').output,
                base.get_layer('conv5_block3_out').output
            ]
        )

    # 5. 构建完整 PraNet 模型，动态 resize 输出
    def build_pranet(input_shape=(288, 384, 3), channel=32):
        inputs   = Input(shape=input_shape)
        c2, c3, c4, c5 = build_backbone(input_shape)(inputs)

        r2 = RFB_modified(512, channel)(c2)
        r3 = RFB_modified(1024, channel)(c3)
        r4 = RFB_modified(2048, channel)(c4)
        agg = Aggregation(channel)(r4, r3, r2)

        ra4 = ReverseAttention(2048)(c4, agg)
        ra3 = ReverseAttention(1024)(c3, ra4)
        ra2 = ReverseAttention(512,)(c2, ra3)

        # 动态 resize：直接将所有输出对齐到原始输入大小
        target_size = tf.shape(inputs)[1:3]
        agg_up = tf.image.resize(agg, target_size, method='bilinear')
        ra4_up = tf.image.resize(ra4, target_size, method='bilinear')
        ra3_up = tf.image.resize(ra3, target_size, method='bilinear')
        ra2_up = tf.image.resize(ra2, target_size, method='bilinear')

        outputs = [
            tf.keras.activations.sigmoid(ra2_up)
        ]

        return models.Model(inputs=inputs, outputs=outputs)

    def build_model():
        return build_pranet(input_shape=(288, 384, 3), channel=32)
    return build_model()



def build_transunet_model():
    # Architecture copied verbatim from TransUNet-Raw-vs-FMM-Train-Inference.ipynb.
    import tensorflow as tf
    from tensorflow.keras import layers, models

    # =========================================================
    # 1) Transformer 相关模块
    # =========================================================

    class AddPositionEmbs(layers.Layer):
        def build(self, input_shape):
            # input_shape: [B, N, C]
            self.pos_emb = self.add_weight(
                name="pos_embedding",
                shape=(1, input_shape[1], input_shape[2]),
                initializer=tf.keras.initializers.RandomNormal(stddev=0.06),
                trainable=True
            )

        def call(self, x):
            return x + self.pos_emb


    class MultiHeadSelfAttention(layers.Layer):
        def __init__(self, hidden_size, num_heads, attn_dropout=0.0):
            super().__init__()
            assert hidden_size % num_heads == 0, "hidden_size must be divisible by num_heads"
            self.hidden_size = hidden_size
            self.num_heads = num_heads
            self.head_dim = hidden_size // num_heads
            self.attn_dropout = layers.Dropout(attn_dropout)

            self.q_dense = layers.Dense(hidden_size)
            self.k_dense = layers.Dense(hidden_size)
            self.v_dense = layers.Dense(hidden_size)
            self.out_dense = layers.Dense(hidden_size)

        def _split_heads(self, x):
            # [B, N, C] -> [B, H, N, D]
            b = tf.shape(x)[0]
            n = tf.shape(x)[1]
            x = tf.reshape(x, [b, n, self.num_heads, self.head_dim])
            return tf.transpose(x, [0, 2, 1, 3])

        def call(self, x, training=None):
            q = self._split_heads(self.q_dense(x))
            k = self._split_heads(self.k_dense(x))
            v = self._split_heads(self.v_dense(x))

            scale = tf.cast(self.head_dim, x.dtype) ** -0.5
            attn = tf.matmul(q, k, transpose_b=True) * scale
            attn = tf.nn.softmax(attn, axis=-1)
            attn = self.attn_dropout(attn, training=training)

            out = tf.matmul(attn, v)                # [B, H, N, D]
            out = tf.transpose(out, [0, 2, 1, 3])  # [B, N, H, D]

            b = tf.shape(out)[0]
            n = tf.shape(out)[1]
            out = tf.reshape(out, [b, n, self.hidden_size])
            out = self.out_dense(out)
            return out

    def gelu_fn(x):
        if hasattr(tf.keras.activations, "gelu"):
            return tf.keras.activations.gelu(x, approximate=False)
        return 0.5 * x * (1.0 + tf.tanh(
            tf.sqrt(2.0 / 3.141592653589793) * (x + 0.044715 * tf.pow(x, 3))
        ))


    class TransformerBlock(layers.Layer):
        def __init__(self, hidden_size, num_heads, mlp_dim, dropout=0.1):
            super().__init__()
            self.norm1 = layers.LayerNormalization(epsilon=1e-6)
            self.attn = MultiHeadSelfAttention(hidden_size, num_heads, attn_dropout=dropout)
            self.drop1 = layers.Dropout(dropout)

            self.norm2 = layers.LayerNormalization(epsilon=1e-6)
            self.mlp = models.Sequential([
                layers.Dense(mlp_dim),
                layers.Lambda(gelu_fn),
                layers.Dropout(dropout),
                layers.Dense(hidden_size),
                layers.Dropout(dropout),
            ])

        def call(self, x, training=None):
            x = x + self.drop1(self.attn(self.norm1(x), training=training), training=training)
            x = x + self.mlp(self.norm2(x), training=training)
            return x


    # =========================================================
    # 2) Decoder 相关模块
    # =========================================================

    class Conv2DReLU(layers.Layer):
        def __init__(self, filters, kernel_size, strides=1, padding="same", use_bn=True):
            super().__init__()
            self.conv = layers.Conv2D(
                filters=filters,
                kernel_size=kernel_size,
                strides=strides,
                padding=padding,
                use_bias=not use_bn,
                kernel_initializer="he_normal"
            )
            self.bn = layers.BatchNormalization() if use_bn else None
            self.relu = layers.ReLU()

        def call(self, x, training=None):
            x = self.conv(x)
            if self.bn is not None:
                x = self.bn(x, training=training)
            x = self.relu(x)
            return x


    class DecoderBlock(layers.Layer):
        def __init__(self, filters):
            super().__init__()
            self.up = layers.UpSampling2D(size=2, interpolation="bilinear")
            self.conv1 = Conv2DReLU(filters, 3)
            self.conv2 = Conv2DReLU(filters, 3)

        def call(self, x, skip=None, training=None):
            x = self.up(x)
            if skip is not None:
                x = tf.concat([x, skip], axis=-1)
            x = self.conv1(x, training=training)
            x = self.conv2(x, training=training)
            return x


    class DecoderCup(layers.Layer):
        def __init__(self, decoder_channels=(256, 128, 64, 16), n_skip=3):
            super().__init__()
            self.n_skip = n_skip
            self.conv_more = Conv2DReLU(512, 3)
            self.blocks = [DecoderBlock(ch) for ch in decoder_channels]

        def call(self, hidden_states, features=None, training=None):
            x = self.conv_more(hidden_states, training=training)
            for i, block in enumerate(self.blocks):
                skip = features[i] if (features is not None and i < self.n_skip) else None
                x = block(x, skip=skip, training=training)
            return x


    class SegmentationHead(layers.Layer):
        def __init__(self, num_classes=1, kernel_size=1, final_act=None):
            super().__init__()
            self.conv = layers.Conv2D(num_classes, kernel_size, padding="same")
            self.final_act = layers.Activation(final_act) if final_act is not None else None

        def call(self, x):
            x = self.conv(x)
            if self.final_act is not None:
                x = self.final_act(x)
            return x


    # =========================================================
    # 3) Backbone
    #    与你上传的 TransUNet-tf 思路一致：ResNet50V2 做 hybrid encoder
    # =========================================================

    def build_backbone(input_shape=(288, 384, 3), weights="imagenet", freeze_backbone=True):
        inputs = layers.Input(shape=input_shape)

        base = tf.keras.applications.ResNet50V2(
            include_top=False,
            weights=weights,
            input_tensor=inputs
        )

        # 和原仓库取 skip 的思路一致：
        # conv3_block4_preact_relu, conv2_block3_preact_relu, conv1_conv
        # transformer 输入来自 conv4_block6_preact_relu
        c1 = base.get_layer("conv1_conv").output                  # /2
        c2 = base.get_layer("conv2_block3_preact_relu").output    # /4
        c3 = base.get_layer("conv3_block4_preact_relu").output    # /8
        c4 = base.get_layer("conv4_block6_preact_relu").output    # /16

        backbone = models.Model(inputs=inputs, outputs=[c1, c2, c3, c4], name="resnet50v2_backbone")

        if freeze_backbone:
            backbone.trainable = False

        return backbone


    # =========================================================
    # 4) 构建完整 TransUNet
    # =========================================================

    def build_transunet(
        input_shape=(288, 384, 3),
        num_classes=1,
        final_act="sigmoid",
        hidden_size=768,
        num_heads=12,
        mlp_dim=3072,
        num_transformer_layers=12,
        transformer_dropout=0.1,
        decoder_channels=(256, 128, 64, 16),
        n_skip=3,
        backbone_weights="imagenet",
        freeze_backbone=True,
    ):
        inputs = layers.Input(shape=input_shape)

        # Backbone features
        c1, c2, c3, c4 = build_backbone(
            input_shape=input_shape,
            weights=backbone_weights,
            freeze_backbone=freeze_backbone
        )(inputs)

        # skip features 顺序和 decoder 对齐：高层 -> 低层
        features = [c3, c2, c1]

        # -----------------------------------------------------
        # Patch Embedding
        # 这里直接把 c4 每个空间位置映射成 token
        # 对于 288x384 输入，c4 一般是 18x24
        # -----------------------------------------------------
        x = layers.Conv2D(
            filters=hidden_size,
            kernel_size=1,
            strides=1,
            padding="same",
            name="embedding"
        )(c4)

        h = x.shape[1]
        w = x.shape[2]
        if h is None or w is None:
            raise ValueError("Static spatial shape is required. Please provide a fixed input_shape.")

        x = layers.Reshape((h * w, hidden_size))(x)
        x = AddPositionEmbs()(x)
        x = layers.Dropout(transformer_dropout)(x)

        # -----------------------------------------------------
        # Transformer Encoder
        # -----------------------------------------------------
        for _ in range(num_transformer_layers):
            x = TransformerBlock(
                hidden_size=hidden_size,
                num_heads=num_heads,
                mlp_dim=mlp_dim,
                dropout=transformer_dropout
            )(x)

        x = layers.LayerNormalization(epsilon=1e-6)(x)

        # token -> feature map
        x = layers.Reshape((h, w, hidden_size))(x)

        # -----------------------------------------------------
        # Decoder
        # -----------------------------------------------------
        x = DecoderCup(
            decoder_channels=decoder_channels,
            n_skip=n_skip
        )(x, features=features)

        # -----------------------------------------------------
        # Segmentation Head
        # -----------------------------------------------------
        x = SegmentationHead(
            num_classes=num_classes,
            kernel_size=1,
            final_act=None
        )(x)

        # 最后 resize 回原图大小
        x = layers.Lambda(
            lambda t: tf.image.resize(t[0], tf.shape(t[1])[1:3], method="bilinear"),
            name="final_resize"
        )([x, inputs])

        if final_act is not None:
            x = layers.Activation(final_act)(x)

        return models.Model(inputs=inputs, outputs=[x], name="TransUNet")


    def build_model():
        # 严格沿用原TransUNet.ipynb实际训练与历史权重对应的轻量配置。
        return build_transunet(
            input_shape=(288, 384, 3),
            num_classes=1,
            final_act="sigmoid",
            hidden_size=384,
            num_heads=4,
            mlp_dim=1536,
            num_transformer_layers=4,
            decoder_channels=(256, 128, 64, 16),
            n_skip=3,
            backbone_weights="imagenet" if RUN_TRAINING else None,
            freeze_backbone=True,
        )
    return build_model()



def build_polyp_pvt():
    # Architecture copied verbatim from Polyp-PVT_Raw-vs-FMM_TensorFlow.ipynb.
    try:
        from keras_cv_attention_models import pvt
    except ImportError as error:
        raise ImportError(
            "缺少TensorFlow PVT实现。请在独立TensorFlow环境安装："
            "pip install keras-cv-attention-models"
        ) from error


    def basic_conv(x, filters, kernel_size=3, name="conv", activation=True):
        """Conv-BN-(ReLU)基本模块，采用channels_last TensorFlow布局。"""
        x = tf.keras.layers.Conv2D(filters, kernel_size, padding="same", use_bias=False,
                                   name=name + "_conv")(x)
        x = tf.keras.layers.BatchNormalization(name=name + "_bn")(x)
        if activation:
            x = tf.keras.layers.ReLU(name=name + "_relu")(x)
        return x


    def channel_attention(x, ratio=16, name="cim_ca"):
        """CIM通道注意力：平均池化和最大池化共享两层1×1映射。"""
        channels = int(x.shape[-1])
        hidden = max(1, channels // ratio)
        reduce_layer = tf.keras.layers.Conv2D(hidden, 1, activation="relu", use_bias=False,
                                               name=name + "_reduce")
        expand_layer = tf.keras.layers.Conv2D(channels, 1, use_bias=False, name=name + "_expand")
        average = expand_layer(reduce_layer(tf.keras.layers.GlobalAveragePooling2D(keepdims=True)(x)))
        maximum = expand_layer(reduce_layer(tf.keras.layers.GlobalMaxPooling2D(keepdims=True)(x)))
        return tf.keras.layers.Activation("sigmoid", name=name + "_gate")(average + maximum)


    def spatial_attention(x, name="cim_sa"):
        """CIM空间注意力：沿通道求平均和最大值后使用7×7卷积生成门控。"""
        average = tf.reduce_mean(x, axis=-1, keepdims=True)
        maximum = tf.reduce_max(x, axis=-1, keepdims=True)
        merged = tf.keras.layers.Concatenate(name=name + "_concat")([average, maximum])
        return tf.keras.layers.Conv2D(1, 7, padding="same", activation="sigmoid",
                                      use_bias=False, name=name + "_gate")(merged)


    def resize_like(x, reference, name):
        """双线性缩放到参考特征的空间尺寸，替代固定倍率以支持384×288。"""
        return tf.keras.layers.Lambda(
            lambda tensors: tf.image.resize(tensors[0], tf.shape(tensors[1])[1:3], method="bilinear"),
            name=name,
        )([x, reference])


    def cascaded_fusion(x4, x3, x2, channel=32):
        """移植Polyp-PVT CFM：乘法筛选高层语义，再逐级拼接融合。"""
        x4_up_x3 = basic_conv(resize_like(x4, x3, "cfm_x4_to_x3"), channel, name="cfm_u1")
        x3_1 = tf.keras.layers.Multiply(name="cfm_mul_x3")([x4_up_x3, x3])
        x4_up_x2 = basic_conv(resize_like(x4, x2, "cfm_x4_to_x2"), channel, name="cfm_u2")
        x3_up_x2 = basic_conv(resize_like(x3, x2, "cfm_x3_to_x2"), channel, name="cfm_u3")
        x2_1 = tf.keras.layers.Multiply(name="cfm_mul_x2")([x4_up_x2, x3_up_x2, x2])
        x3_2 = tf.keras.layers.Concatenate(name="cfm_concat_x3")([
            x3_1, basic_conv(resize_like(x4, x3, "cfm_x4_to_x3_b"), channel, name="cfm_u4")
        ])
        x3_2 = basic_conv(x3_2, 2 * channel, name="cfm_concat2")
        x2_2 = tf.keras.layers.Concatenate(name="cfm_concat_x2")([
            x2_1, basic_conv(resize_like(x3_2, x2, "cfm_x3_2_to_x2"), 2 * channel, name="cfm_u5")
        ])
        x2_2 = basic_conv(x2_2, 3 * channel, name="cfm_concat3")
        return basic_conv(x2_2, channel, name="cfm_output")


    def similarity_aggregation(feature, cim_feature, channel=32, state_channels=16, mids=4):
        """按官方张量关系移植SAM的区域先验、图卷积和反投影。

        feature为CFM的1/8尺度特征；cim_feature为1/4尺度浅层定位特征。SAM先将
        CIM压缩到32通道并缩放到CFM尺寸，用softmax后的第2个通道作为空间先验；
        随后在4×4共16个图节点中传播关系，最后反投影到每个空间像素并残差相加。
        """
        edge = basic_conv(cim_feature, channel, 1, name="sam_edge_transform")
        edge = resize_like(edge, feature, "sam_edge_resize")
        edge = tf.keras.layers.Lambda(
            lambda tensor: tf.nn.softmax(tensor, axis=-1)[..., 1:2],
            name="sam_edge_softmax_channel1",
        )(edge)
        state = tf.keras.layers.Conv2D(state_channels, 1, name="sam_state")(feature)
        projection = tf.keras.layers.Conv2D(state_channels, 1, name="sam_projection")(feature)
        masked_projection = tf.keras.layers.Multiply(name="sam_mask_projection")([projection, edge])
        # 原实现用ResizeArea模拟自适应平均池化，但部分TensorFlow版本没有为
        # ResizeArea注册反向梯度。当前固定输入下该特征为36×48，目标为6×6，
        # 因而6×8平均池化与自适应平均池化完全等价，并且可正常反向传播。
        target_size = mids + 2
        feature_height = int(masked_projection.shape[1])
        feature_width = int(masked_projection.shape[2])
        if feature_height % target_size or feature_width % target_size:
            raise ValueError(
                "SAM特征尺寸必须能整除目标池化尺寸：%dx%d -> %dx%d" % (
                    feature_height, feature_width, target_size, target_size
                )
            )
        pool_size = (feature_height // target_size, feature_width // target_size)
        anchors = tf.keras.layers.AveragePooling2D(
            pool_size=pool_size, strides=pool_size, padding="valid",
            name="sam_anchor_average_pool",
        )(masked_projection)
        anchors = tf.keras.layers.Cropping2D(
            cropping=((1, 1), (1, 1)), name="sam_anchor_crop"
        )(anchors)

        def project_to_graph(tensors):
            current_state, current_projection, current_anchors = tensors
            batch = tf.shape(current_state)[0]
            height, width = tf.shape(current_state)[1], tf.shape(current_state)[2]
            state_flat = tf.transpose(tf.reshape(current_state, [batch, height * width, state_channels]), [0, 2, 1])
            projection_flat = tf.transpose(
                tf.reshape(current_projection, [batch, height * width, state_channels]), [0, 2, 1]
            )
            anchor_flat = tf.transpose(
                tf.reshape(current_anchors, [batch, mids * mids, state_channels]), [0, 2, 1]
            )
            assignment = tf.matmul(anchor_flat, projection_flat, transpose_a=True)
            assignment = tf.nn.softmax(assignment, axis=1)
            graph_state = tf.matmul(state_flat, assignment, transpose_b=True)
            return graph_state, assignment

        graph_state, assignment = tf.keras.layers.Lambda(
            project_to_graph, name="sam_project_to_graph"
        )([state, projection, anchors])
        # 对应官方GCN：先沿节点通道卷积并做残差，再沿state通道卷积。
        graph_h = tf.keras.layers.Conv1D(mids * mids, 1, name="sam_gcn_nodes")(graph_state)
        graph_h = tf.keras.layers.Subtract(name="sam_gcn_subtract")([graph_h, graph_state])
        graph_h = tf.keras.layers.ReLU(name="sam_gcn_relu")(graph_h)
        graph_h_t = tf.keras.layers.Permute((2, 1), name="sam_gcn_transpose1")(graph_h)
        graph_h_t = tf.keras.layers.Conv1D(state_channels, 1, name="sam_gcn_states")(graph_h_t)
        graph_relation = tf.keras.layers.Permute((2, 1), name="sam_gcn_transpose2")(graph_h_t)

        def reverse_project(tensors):
            relation, assign, reference = tensors
            batch = tf.shape(reference)[0]
            h, w = tf.shape(reference)[1], tf.shape(reference)[2]
            spatial = tf.matmul(relation, assign)
            return tf.reshape(tf.transpose(spatial, [0, 2, 1]), [batch, h, w, state_channels])

        spatial_state = tf.keras.layers.Lambda(
            reverse_project, name="sam_reverse_project"
        )([graph_relation, assignment, feature])
        extended = tf.keras.layers.Conv2D(channel, 1, use_bias=False, name="sam_extend")(spatial_state)
        return tf.keras.layers.Add(name="sam_residual")([feature, extended])


    def discover_pvt_stages(backbone, input_height, input_width):
        """按输出空间尺寸发现四个PVT stage，避免依赖易变化的第三方层名。"""
        expected = {
            (input_height // 4, input_width // 4): "stage1",
            (input_height // 8, input_width // 8): "stage2",
            (input_height // 16, input_width // 16): "stage3",
            (input_height // 32, input_width // 32): "stage4",
        }
        selected = {}
        for layer in backbone.layers:
            output = getattr(layer, "output", None)
            shape = getattr(output, "shape", None)
            if shape is None or len(shape) != 4 or shape[1] is None or shape[2] is None:
                continue
            spatial = (int(shape[1]), int(shape[2]))
            if spatial in expected:
                selected[expected[spatial]] = output  # 同尺度取最后一层，接近stage末端
        missing = [name for name in expected.values() if name not in selected]
        if missing:
            available = [(layer.name, getattr(getattr(layer, "output", None), "shape", None))
                         for layer in backbone.layers]
            raise RuntimeError("无法发现PVT多尺度输出：%s；可用层：%s" % (missing, available[-30:]))
        return [selected["stage1"], selected["stage2"], selected["stage3"], selected["stage4"]]

    def build_model():
        """只替换母版中的U-Net+InceptionV3模型，返回未编译的TensorFlow Polyp-PVT。

        输入、Raw/FMM读取、Train/Val划分、损失函数、优化器、回调、权重命名、推理、
        指标和可视化全部由后续母版代码负责。本函数仅负责构建网络结构。
        """
        if RUN_TRAINING and not PVT_PRETRAINED_WEIGHT.is_file():
            raise FileNotFoundError(
                "找不到手动下载的PVTv2-B2 ImageNet权重：%s\n"
                "请将pvt_v2_b2_imagenet.h5放入该位置。" % PVT_PRETRAINED_WEIGHT
            )
        backbone = pvt.PVT_V2B2(
            input_shape=(IMAGE_HEIGHT, IMAGE_WIDTH, 3),
            num_classes=0,
            # 直接传入本地文件，避免服务器或Windows环境再次访问GitHub。
            pretrained=str(PVT_PRETRAINED_WEIGHT) if RUN_TRAINING else None,
        )
        x1, x2, x3, x4 = discover_pvt_stages(backbone, IMAGE_HEIGHT, IMAGE_WIDTH)

        # CIM：用最低层的细节特征生成通道注意力和空间注意力。
        x1_cim = tf.keras.layers.Multiply(name="cim_channel_mul")([
            x1, channel_attention(x1)
        ])
        cim = tf.keras.layers.Multiply(name="cim_spatial_mul")([
            x1_cim, spatial_attention(x1_cim)
        ])

        # CFM：将PVT编码器的1/8、1/16和1/32高层特征统一到32通道并逐级融合。
        x2_t = basic_conv(x2, 32, 1, name="trans_x2")
        x3_t = basic_conv(x3, 32, 1, name="trans_x3")
        x4_t = basic_conv(x4, 32, 1, name="trans_x4")
        cfm = cascaded_fusion(x4_t, x3_t, x2_t, channel=32)

        # SAM：利用CIM浅层定位信息细化CFM的语义分割特征。
        sam = similarity_aggregation(cfm, cim, channel=32)
        logits = tf.keras.layers.Conv2D(1, 1, name="segmentation_logits")(sam)
        logits = tf.keras.layers.Lambda(
            lambda tensor: tf.image.resize(
                tensor, (IMAGE_HEIGHT, IMAGE_WIDTH), method="bilinear"
            ),
            name="full_resolution",
        )(logits)
        probability = tf.keras.layers.Activation(
            "sigmoid", dtype="float32", name="probability"
        )(logits)
        return tf.keras.Model(backbone.input, probability, name="PolypPVT_TensorFlow")
    return build_model()



MODEL_BUILDERS = {
    'U-Net++': build_u_net_plus_plus,
    'U-Net+V3': build_u_net_v3,
    'FPN+V3': build_fpn_v3,
    'LinkNet+V3': build_linknet_v3,
    'PSPNet+V3': build_pspnet_v3,
    'SegNet-Simple': build_segnet_simple,
    'SegNet': build_segnet,
    'PraNet': build_pranet_model,
    'TransUNet': build_transunet_model,
    'Polyp-PVT': build_polyp_pvt,
}


In [8]:
# =========================
# Timing helpers
# =========================
def make_inference_function(model):
    @tf.function(
        input_signature=[
            tf.TensorSpec(
                shape=(BATCH_SIZE, IMAGE_HEIGHT, IMAGE_WIDTH, 3),
                dtype=tf.float32,
            )
        ]
    )
    def infer(batch):
        prediction = model(batch, training=False)
        if isinstance(prediction, (list, tuple)):
            prediction = prediction[0]
        return prediction
    return infer


def timed_pipeline(decoded_bgr, decoded_gray, condition, infer):
    total_started = time.perf_counter()

    preprocess_started = total_started
    if condition == "Raw":
        image = preprocess_raw(decoded_bgr)
    elif condition == "FMM-99.5":
        image = preprocess_fmm(decoded_bgr, decoded_gray)
    else:
        raise ValueError(f"Unknown condition: {condition}")
    preprocess_finished = time.perf_counter()

    inference_started = preprocess_finished
    batch = tf.convert_to_tensor(image[None, ...], dtype=tf.float32)
    prediction = infer(batch).numpy()  # Materialization synchronizes GPU execution.
    inference_finished = time.perf_counter()

    preprocess_ms = (preprocess_finished - preprocess_started) * 1000.0
    inference_ms = (inference_finished - inference_started) * 1000.0
    total_ms = (inference_finished - total_started) * 1000.0

    if prediction.shape != (BATCH_SIZE, IMAGE_HEIGHT, IMAGE_WIDTH, 1):
        raise ValueError(f"Unexpected prediction shape: {prediction.shape}")
    if not np.isfinite(prediction).all():
        raise FloatingPointError("Prediction contains NaN or Inf.")
    return preprocess_ms, inference_ms, total_ms


def warm_up(condition, infer, paths):
    for image_index, image_path in enumerate(paths):
        color, gray = decode_image_pair(image_path)
        max_attempts = 2 if image_index == 0 else 1

        for attempt in range(max_attempts):
            try:
                timed_pipeline(color, gray, condition, infer)
                break
            except FloatingPointError:
                if attempt + 1 == max_attempts:
                    raise
                warnings.warn(
                    f"{condition}: non-finite output during the first "
                    "unmeasured PTX warm-up; retrying once."
                )


def benchmark_condition(model_name, condition, infer, image_frame):
    warmup_count = min(EFFECTIVE_WARMUP, len(image_frame))
    warm_up(condition, infer, image_frame["image_path"].iloc[:warmup_count])

    rows = []
    for run_index in range(1, EFFECTIVE_RUNS + 1):
        run_started = time.perf_counter()
        for order, row in enumerate(image_frame.itertuples(index=False), start=1):
            color, gray = decode_image_pair(row.image_path)  # Disk I/O excluded.
            preprocess_ms, inference_ms, total_ms = timed_pipeline(
                color, gray, condition, infer
            )
            rows.append(
                {
                    "model": model_name,
                    "condition": condition,
                    "run_index": run_index,
                    "image_order": order,
                    "image_name": row.image_name,
                    "preprocess_ms": preprocess_ms,
                    "inference_ms": inference_ms,
                    "total_ms": total_ms,
                }
            )
        elapsed = time.perf_counter() - run_started
        print(
            f"{model_name} | {condition} | run {run_index}/{EFFECTIVE_RUNS} "
            f"completed in {elapsed:.1f} s"
        )
    return rows


## Execute benchmark

The next cell processes one model at a time. It overwrites that model's partial CSV only after both Raw and FMM conditions finish, then clears the TensorFlow session before moving to the next architecture. This permits a failed model to be rerun by setting `RUN_MODELS` to a shorter list without repeating completed architectures.


In [9]:
completed_partials = []
for model_name in RUN_MODELS:
    print("\n" + "=" * 80)
    print("Building:", model_name)
    tf.keras.backend.clear_session()
    gc.collect()

    model = MODEL_BUILDERS[model_name]()
    infer = make_inference_function(model)
    raw_weight, fmm_weight = resolve_weight_paths(model_name)
    model_rows = []

    for condition, weight_path in (
        ("Raw", raw_weight),
        ("FMM-99.5", fmm_weight),
    ):
        load_selected_weights(model, weight_path, model_name)
        condition_rows = benchmark_condition(
            model_name, condition, infer, active_manifest
        )
        model_rows.extend(condition_rows)

    model_frame = pd.DataFrame(model_rows)
    expected_model_rows = len(active_manifest) * EFFECTIVE_RUNS * 2
    if len(model_frame) != expected_model_rows:
        raise AssertionError(
            f"{model_name}: expected {expected_model_rows} rows, "
            f"found {len(model_frame)}"
        )
    partial_path = PARTIAL_DIR / (
        model_name.lower().replace("+", "_plus_").replace("-", "_")
        + ".csv"
    )
    model_frame.to_csv(partial_path, index=False)
    completed_partials.append(partial_path)
    print("Saved:", partial_path)

    del model_frame, model_rows, model, infer
    tf.keras.backend.clear_session()
    gc.collect()

print(f"Completed {len(completed_partials)} model partials.")



Building: U-Net++
原始U-Net++（2018 CVPR版本）


2026-11-09 16:39:43.492062: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1943] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.


U-Net++: strictly loaded legacy HDF5 weights by topology


2026-11-09 16:47:08.375462: W tensorflow/stream_executor/gpu/asm_compiler.cc:80] Couldn't get ptxas version string: INTERNAL: Couldn't invoke ptxas --version
2026-11-09 16:47:08.376130: W tensorflow/stream_executor/gpu/redzone_allocator.cc:314] INTERNAL: Failed to launch ptxas
Relying on driver to perform ptx compilation. 
Modify $PATH to customize ptxas location.
This message will be only logged once.


U-Net++ | Raw | run 1/3 completed in 61.7 s
U-Net++ | Raw | run 2/3 completed in 61.7 s
U-Net++ | Raw | run 3/3 completed in 61.7 s
U-Net++: strictly loaded legacy HDF5 weights by topology
U-Net++ | FMM-99.5 | run 1/3 completed in 111.0 s
U-Net++ | FMM-99.5 | run 2/3 completed in 110.9 s
U-Net++ | FMM-99.5 | run 3/3 completed in 111.4 s
Saved: /home/wujieyu/jupyterlab/tensorflow/supplement/PR-R2/code/Supplementary_Code/03_runtime_analysis/outputs_end_to_end_runtime/per_model/u_net_plus__plus_.csv

Building: U-Net+V3
U-Net+V3: strictly loaded legacy HDF5 weights by topology
U-Net+V3 | Raw | run 1/3 completed in 58.2 s
U-Net+V3 | Raw | run 2/3 completed in 58.1 s
U-Net+V3 | Raw | run 3/3 completed in 58.1 s
U-Net+V3: strictly loaded legacy HDF5 weights by topology
U-Net+V3 | FMM-99.5 | run 1/3 completed in 107.7 s
U-Net+V3 | FMM-99.5 | run 2/3 completed in 108.4 s
U-Net+V3 | FMM-99.5 | run 3/3 completed in 107.9 s
Saved: /home/wujieyu/jupyterlab/tensorflow/supplement/PR-R2/code/Supplemen

/tmp/ipykernel_2077611/2470751476.py:61: UserWarning: Raw: non-finite output during the first unmeasured PTX warm-up; retrying once.
  warnings.warn(


PraNet | Raw | run 1/3 completed in 66.8 s
PraNet | Raw | run 2/3 completed in 67.2 s
PraNet | Raw | run 3/3 completed in 67.5 s
PraNet: strictly loaded legacy HDF5 weights by topology
PraNet | FMM-99.5 | run 1/3 completed in 117.7 s
PraNet | FMM-99.5 | run 2/3 completed in 117.9 s
PraNet | FMM-99.5 | run 3/3 completed in 117.6 s
Saved: /home/wujieyu/jupyterlab/tensorflow/supplement/PR-R2/code/Supplementary_Code/03_runtime_analysis/outputs_end_to_end_runtime/per_model/pranet.csv

Building: TransUNet
TransUNet: strictly loaded legacy HDF5 weights by topology
TransUNet | Raw | run 1/3 completed in 47.7 s
TransUNet | Raw | run 2/3 completed in 47.6 s
TransUNet | Raw | run 3/3 completed in 47.3 s
TransUNet: strictly loaded legacy HDF5 weights by topology
TransUNet | FMM-99.5 | run 1/3 completed in 97.1 s
TransUNet | FMM-99.5 | run 2/3 completed in 96.6 s
TransUNet | FMM-99.5 | run 3/3 completed in 97.2 s
Saved: /home/wujieyu/jupyterlab/tensorflow/supplement/PR-R2/code/Supplementary_Code/03

In [10]:
# =========================
# Aggregate and export
# =========================
def partial_path_for(model_name):
    filename = model_name.lower().replace("+", "_plus_").replace("-", "_") + ".csv"
    return PARTIAL_DIR / filename


expected_models = RUN_MODELS if DRY_RUN else ALL_MODELS
missing_partials = [
    model_name
    for model_name in expected_models
    if not partial_path_for(model_name).is_file()
]
if missing_partials:
    raise FileNotFoundError(f"Missing model partials: {missing_partials}")

detail = pd.concat(
    [pd.read_csv(partial_path_for(model_name)) for model_name in expected_models],
    ignore_index=True,
)
expected_rows = len(expected_models) * len(active_manifest) * EFFECTIVE_RUNS * 2
if len(detail) != expected_rows:
    raise AssertionError(f"Expected {expected_rows} rows, found {len(detail)}")

expected_per_group = len(active_manifest) * EFFECTIVE_RUNS
group_counts = detail.groupby(["model", "condition"]).size()
if not (group_counts == expected_per_group).all():
    raise AssertionError(f"Incomplete model/condition groups:\n{group_counts}")

# Run-level means: one observation per complete dataset run.
run_level = (
    detail.groupby(["model", "condition", "run_index"], as_index=False)[
        ["preprocess_ms", "inference_ms", "total_ms"]
    ]
    .mean()
)

# Per-image average across runs, used for median and P95.
per_image_average = (
    detail.groupby(["model", "condition", "image_name"], as_index=False)[
        ["preprocess_ms", "inference_ms", "total_ms"]
    ]
    .mean()
)

def mean_and_sd(values):
    mean = float(values.mean())
    sd = float(values.std(ddof=1)) if len(values) > 1 else 0.0
    return mean, sd


def format_mean_sd(mean, sd):
    return f"{mean:.2f} ± {sd:.2f}"


summary_rows = []
for model_name in expected_models:
    raw_runs = run_level[
        (run_level["model"] == model_name)
        & (run_level["condition"] == "Raw")
    ].sort_values("run_index")
    fmm_runs = run_level[
        (run_level["model"] == model_name)
        & (run_level["condition"] == "FMM-99.5")
    ].sort_values("run_index")
    fmm_images = per_image_average[
        (per_image_average["model"] == model_name)
        & (per_image_average["condition"] == "FMM-99.5")
    ]
    paired_totals = raw_runs[["run_index", "total_ms"]].merge(
        fmm_runs[["run_index", "total_ms"]],
        on="run_index",
        suffixes=("_raw", "_fmm"),
        validate="one_to_one",
    )
    if len(paired_totals) != EFFECTIVE_RUNS:
        raise AssertionError(f"{model_name}: incomplete paired runtime runs")
    added_latency = (
        paired_totals["total_ms_fmm"] - paired_totals["total_ms_raw"]
    )

    raw_inference_mean, raw_inference_sd = mean_and_sd(raw_runs["inference_ms"])
    raw_total_mean, raw_total_sd = mean_and_sd(raw_runs["total_ms"])
    fmm_preprocess_mean, fmm_preprocess_sd = mean_and_sd(
        fmm_runs["preprocess_ms"]
    )
    fmm_inference_mean, fmm_inference_sd = mean_and_sd(
        fmm_runs["inference_ms"]
    )
    fmm_total_mean, fmm_total_sd = mean_and_sd(fmm_runs["total_ms"])
    added_mean, added_sd = mean_and_sd(added_latency)

    summary_rows.append(
        {
            "Model": model_name,
            "Images": len(active_manifest),
            "Runs": EFFECTIVE_RUNS,
            "Raw segmentation inference mean (ms)": raw_inference_mean,
            "Raw segmentation inference SD (ms)": raw_inference_sd,
            "Raw pipeline mean (ms)": raw_total_mean,
            "Raw pipeline SD (ms)": raw_total_sd,
            "FMM preprocessing mean (ms)": fmm_preprocess_mean,
            "FMM preprocessing SD (ms)": fmm_preprocess_sd,
            "FMM segmentation inference mean (ms)": fmm_inference_mean,
            "FMM segmentation inference SD (ms)": fmm_inference_sd,
            "FMM pipeline mean (ms)": fmm_total_mean,
            "FMM pipeline SD (ms)": fmm_total_sd,
            "Added latency mean (ms)": added_mean,
            "Added latency SD (ms)": added_sd,
            "FMM pipeline median (ms)": float(fmm_images["total_ms"].median()),
            "FMM pipeline P95 (ms)": float(
                np.percentile(fmm_images["total_ms"], 95)
            ),
            "FMM throughput (FPS)": 1000.0 / fmm_total_mean,
        }
    )
summary = pd.DataFrame(summary_rows)

supplementary_table = pd.DataFrame(
    {
        "Model": summary["Model"],
        "Raw segmentation inference, mean ± SD (ms)": [
            format_mean_sd(mean, sd)
            for mean, sd in zip(
                summary["Raw segmentation inference mean (ms)"],
                summary["Raw segmentation inference SD (ms)"],
            )
        ],
        "Raw pipeline, mean ± SD (ms)": [
            format_mean_sd(mean, sd)
            for mean, sd in zip(
                summary["Raw pipeline mean (ms)"],
                summary["Raw pipeline SD (ms)"],
            )
        ],
        "FMM preprocessing, mean ± SD (ms)": [
            format_mean_sd(mean, sd)
            for mean, sd in zip(
                summary["FMM preprocessing mean (ms)"],
                summary["FMM preprocessing SD (ms)"],
            )
        ],
        "FMM segmentation inference, mean ± SD (ms)": [
            format_mean_sd(mean, sd)
            for mean, sd in zip(
                summary["FMM segmentation inference mean (ms)"],
                summary["FMM segmentation inference SD (ms)"],
            )
        ],
        "FMM pipeline, mean ± SD (ms)": [
            format_mean_sd(mean, sd)
            for mean, sd in zip(
                summary["FMM pipeline mean (ms)"],
                summary["FMM pipeline SD (ms)"],
            )
        ],
        "Added latency, mean ± SD (ms)": [
            format_mean_sd(mean, sd)
            for mean, sd in zip(
                summary["Added latency mean (ms)"],
                summary["Added latency SD (ms)"],
            )
        ],
        "FMM pipeline median (ms)": summary[
            "FMM pipeline median (ms)"
        ].round(2),
        "FMM pipeline P95 (ms)": summary["FMM pipeline P95 (ms)"].round(2),
        "FMM throughput (FPS)": summary["FMM throughput (FPS)"].round(2),
    }
)

supplementary_caption = (
    "End-to-end runtime of the Raw and FMM-99.5 segmentation pipelines."
)
supplementary_note = (
    "Values reported as mean ± SD were calculated across three complete runs "
    "over 2,987 images. Median and P95 were calculated after averaging the "
    "three measurements for each image. Batch size was 1. Disk I/O and model "
    "loading were excluded. Segmentation inference includes CPU-to-GPU tensor "
    "conversion and synchronized GPU inference. Raw pipeline denotes ordinary "
    "model preprocessing plus segmentation inference; FMM pipeline denotes "
    "highlight detection, mask scaling, FMM restoration, model preprocessing, "
    "and segmentation inference. Added latency is the paired per-run difference "
    "between the FMM and Raw pipeline means."
)

# Directly measured totals must closely match the sum of the measured stages.
timing_residual = np.abs(
    detail["total_ms"]
    - (detail["preprocess_ms"] + detail["inference_ms"])
)
if float(timing_residual.max()) > 0.1:
    raise AssertionError(
        f"Unexpected timing residual: max={timing_residual.max():.6f} ms"
    )

manifest_export = manifest[["image_name", "file_size_bytes", "sha256"]].copy()
environment = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "mode": "dry_run" if DRY_RUN else "formal",
    "platform": platform.platform(),
    "python": sys.version,
    "tensorflow": tf.__version__,
    "opencv": cv2.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "keras_cv_attention_models": kecam_version,
    "openpyxl": importlib_metadata.version("openpyxl"),
    "conda_prefix": os.environ.get("CONDA_PREFIX"),
    "gpu": gpu_query,
    "cpu_threads": CPU_THREADS,
    "batch_size": BATCH_SIZE,
    "input_images": len(active_manifest),
    "runs": EFFECTIVE_RUNS,
    "warmup_images": EFFECTIVE_WARMUP,
    "percentile": PERCENTILE,
    "inpaint_radius": INPAINT_RADIUS,
    "jpeg_quality": JPEG_QUALITY,
    "image_size": [IMAGE_WIDTH, IMAGE_HEIGHT],
    "mask_rebinarized_after_bicubic_resize": False,
    "disk_io_included": False,
    "model_loading_included": False,
    "models": expected_models,
}

manifest_export.to_csv(
    OUTPUT_DIR / "end_to_end_runtime_input_manifest.csv", index=False
)
detail.to_csv(OUTPUT_DIR / "end_to_end_runtime_per_image.csv", index=False)
run_level.to_csv(
    OUTPUT_DIR / "end_to_end_runtime_run_level.csv", index=False
)
supplementary_table.to_csv(
    OUTPUT_DIR / "end_to_end_runtime_summary.csv", index=False
)
summary.to_csv(
    OUTPUT_DIR / "end_to_end_runtime_summary_numeric.csv", index=False
)
(OUTPUT_DIR / "end_to_end_runtime_supplementary_table_note.txt").write_text(
    supplementary_caption + "\nNote. " + supplementary_note,
    encoding="utf-8",
)
excel_path = OUTPUT_DIR / "end_to_end_runtime_summary.xlsx"
if importlib.util.find_spec("openpyxl") is not None:
    excel_engine = "openpyxl"
elif importlib.util.find_spec("xlsxwriter") is not None:
    excel_engine = "xlsxwriter"
else:
    excel_engine = None

if excel_engine is not None:
    with pd.ExcelWriter(excel_path, engine=excel_engine) as writer:
        supplementary_table.to_excel(
            writer, sheet_name="Supplementary table", index=False
        )
        summary.to_excel(writer, sheet_name="Numeric summary", index=False)
        run_level.to_excel(writer, sheet_name="Run-level means", index=False)
        WEIGHT_MANIFEST.to_excel(writer, sheet_name="Weights", index=False)
else:
    warnings.warn(
        "No Excel writer engine is installed; the complete CSV "
        "outputs were saved, but the optional XLSX workbook was skipped."
    )
(OUTPUT_DIR / "end_to_end_runtime_environment.json").write_text(
    json.dumps(environment, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("Exported runtime evidence to:", OUTPUT_DIR)
print("\n" + supplementary_caption)
display(supplementary_table)
print("Note.", supplementary_note)


Exported runtime evidence to: /home/wujieyu/jupyterlab/tensorflow/supplement/PR-R2/code/Supplementary_Code/03_runtime_analysis/outputs_end_to_end_runtime

End-to-end runtime of the Raw and FMM-99.5 segmentation pipelines.


,Model,"Raw segmentation inference, mean ± SD (ms)","Raw pipeline, mean ± SD (ms)","FMM preprocessing, mean ± SD (ms)","FMM segmentation inference, mean ± SD (ms)","FMM pipeline, mean ± SD (ms)","Added latency, mean ± SD (ms)",FMM pipeline median (ms),FMM pipeline P95 (ms),FMM throughput (FPS)
0,U-Net++,13.04 ± 0.02,13.49 ± 0.02,16.73 ± 0.04,13.27 ± 0.04,30.00 ± 0.07,16.52 ± 0.09,22.77,57.12,33.33
1,U-Net+V3,11.85 ± 0.00,12.29 ± 0.00,16.72 ± 0.06,12.22 ± 0.05,28.94 ± 0.11,16.65 ± 0.11,21.54,57.10,34.55
2,FPN+V3,12.99 ± 0.02,13.43 ± 0.03,16.76 ± 0.07,13.34 ± 0.03,30.10 ± 0.05,16.67 ± 0.02,22.66,58.20,33.22
3,LinkNet+V3,9.59 ± 0.02,10.02 ± 0.02,16.65 ± 0.10,10.14 ± 0.08,26.79 ± 0.18,16.77 ± 0.18,19.33,55.19,37.33
4,PSPNet+V3,2.84 ± 0.14,3.27 ± 0.15,16.63 ± 0.06,3.20 ± 0.05,19.83 ± 0.10,16.56 ± 0.18,12.38,48.33,50.43
5,SegNet-Simple,11.11 ± 0.03,11.55 ± 0.03,16.67 ± 0.05,11.29 ± 0.01,27.96 ± 0.05,16.41 ± 0.02,20.74,54.98,35.77
6,SegNet,11.91 ± 0.08,12.35 ± 0.08,16.70 ± 0.05,12.22 ± 0.02,28.91 ± 0.06,16.57 ± 0.12,21.63,56.07,34.59
7,PraNet,14.85 ± 0.10,15.30 ± 0.10,16.80 ± 0.05,15.41 ± 0.10,32.22 ± 0.06,16.92 ± 0.10,24.75,60.45,31.04
8,TransUNet,8.31 ± 0.05,8.74 ± 0.06,16.69 ± 0.03,8.61 ± 0.10,25.30 ± 0.10,16.56 ± 0.14,18.01,52.75,39.52
9,Polyp-PVT,9.46 ± 0.03,9.89 ± 0.03,16.63 ± 0.06,9.72 ± 0.06,26.35 ± 0.01,16.47 ± 0.05,19.15,53.21,37.95


Note. Values reported as mean ± SD were calculated across three complete runs over 2,987 images. Median and P95 were calculated after averaging the three measurements for each image. Batch size was 1. Disk I/O and model loading were excluded. Segmentation inference includes CPU-to-GPU tensor conversion and synchronized GPU inference. Raw pipeline denotes ordinary model preprocessing plus segmentation inference; FMM pipeline denotes highlight detection, mask scaling, FMM restoration, model preprocessing, and segmentation inference. Added latency is the paired per-run difference between the FMM and Raw pipeline means.


## Interpretation and manuscript use

- The existing approximately 2.61 ms value remains a standardized preprocessing-only benchmark and must not be algebraically added to model inference time.
- `FMM pipeline, mean ± SD (ms)` is measured directly from decoded input through synchronized probability-map output.
- For Supplementary Table S22, retain the existing preprocessing comparison as Panel A and use the formal summary above as Panel B.
- Report the RTX 5080 inference hardware separately from the hardware used to train the archived checkpoints.
